# 세부목적 3 · 지형 기복·강수 공간변동성과 GPM–ASOS 편차
**측량학(2) 1조 | 2020~2024년 여름철, 부산·울산·창원**

**연구질문**: 관측지점별 GPM 0.1° 격자의 지형 기복 및 강수 공간변동성과 GPM–ASOS 일강수량 편차는 어떤 관련성을 보이는가?

**구성**: ① 자료 불러오기 → ② 분석일 검증 → ③ 지형지표와 460일 오차 비교 → ④ 100일 공간 CV 및 절대편차 → ⑤ 스피어만 상관 → ⑥ 강수규모/22일 보조 비교 → ⑦ 표·그래프 저장.

**재현 범위와 자료 출처**
- NASA GPM-IMERG V07과 ASOS가 결합된 1,380건의 검증된 공통 마스터 CSV (ASOS 공란을 0mm로 보정한 기존 분석 기준)
- Copernicus DEM GLO-30에서 **이미 산정한** 지점별 지형 통계 요약 CSV
- 기상청 500m 일강수량(`rn_day`)에서 **이미 산정한** 육지 격자 CV 등 100일·300건 CSV
- **이 노트북은 500m 개별 원시 격자값을 다시 다운로드하거나 CV를 원시 픽셀부터 재산출하지 않는다.** 해당 원자료가 제공되지 않아 기존 중간 결과에서 통계·도표를 재현한다. 지형요약도 기존 산출값을 사용한다.
- 아래 CSV들은 노트북 안에 스냅샷으로 내장되어 있어 Colab에서 **모두 실행**하면 데이터 업로드 없이 재계산 가능. GitHub의 `data/processed/`에도 원본 CSV를 별도 공개한다.

자료 설명: KMA 500m 격자값은 각 날짜 **23:55 KST 자료의 격자 내 육지 중심점**에 한정되며, GPM 격자의 면적가중 전체 강수량을 뜻하지 않는다. `rn_day`의 원시자료 단위/집계 시각 해석은 원기관 메타데이터 확인이 별도로 필요하다.


In [ ]:
import sys, platform
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from pathlib import Path
from IPython.display import display
OUT = Path('outputs')
OUT.mkdir(exist_ok=True)
STATIONS = ['부산','울산','창원']
NAME_EN = {'부산':'Busan','울산':'Ulsan','창원':'Changwon'}
print(f'Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__} | matplotlib {matplotlib.__version__}')


In [ ]:
# 보고서에 사용한 원본/중간 산출 CSV를 gzip으로 내장한 스냅샷 (2026-11-26 자료)
# GitHub의 data/processed/에 같은 파일이 있으면 그것을 우선 사용합니다.
import base64, gzip, io
from pathlib import Path
EMBEDDED_DATA = {
    'MASTER_GPM_ASOS_2020_2024_JJA_1380_corrected.csv': (
        '...'
    ),
    'subgoal3_spatial_100days_300rows.csv': (
        '...'
        'AA=='
    ),
    'subgoal3_terrain_summary.csv': (
        'H4sIAAAAAAAC/3WPTUrEQBCF9zlLU6R/032aLFRcSMCFB4jDLBTEiTvRIBEEZ8BFGEVHZPA+SQWvYCeT6uCAvXr91atXVT9f3/icY1WwbnHZfF'
        'TNa9Uu5mnGdqK7KfEp7643WC89HD3rLc5eetvh0XFv9AntVd6u3rA6bz/nWC7T04MzhlX+X6V87CsDTk8ysR8ysL32gek4znbNzXqFDwVe3A4s'
        '9P/FISLgqH3PcVYzLiRYb+mf/xiQgj4SlBu1FJAQNwkIO2qnwMTkicFRkJFgSEtjfDBnQukI7zb9UGXBSLIqiMcMZqeJCWiSToDik3Y2WKZGDY'
        '72k1oykfiVTYT1Fu8LxpVfJ5zlD1YUwR1oRX0WbLiRAyePNeBovJSQkF9L4EngwgdbJmIV/QJFn9nESgIAAA=='
    ),
}
def load_study_csv(filename):
    paths = [Path('data/processed') / filename, Path('../data/processed') / filename, Path(filename)]
    for p in paths:
        if p.is_file():
            print(f'입력 파일: {p}')
            return pd.read_csv(p, encoding='utf-8-sig')
    payload = ''.join(EMBEDDED_DATA[filename])
    raw = gzip.decompress(base64.b64decode(payload))
    print(f'내장된 연구용 CSV 사용: {filename}')
    return pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig')


In [ ]:
master = load_study_csv('MASTER_GPM_ASOS_2020_2024_JJA_1380_corrected.csv')
spatial = load_study_csv('subgoal3_spatial_100days_300rows.csv')
terrain = load_study_csv('subgoal3_terrain_summary.csv')
master['date_kst'] = pd.to_datetime(master['date_kst']).dt.strftime('%Y-%m-%d')
spatial['날짜'] = pd.to_datetime(spatial['날짜']).dt.strftime('%Y-%m-%d')
assert len(master)==1380 and master['date_kst'].nunique()==460
assert len(spatial)==300 and spatial['날짜'].nunique()==100
assert set(terrain['지점'])==set(STATIONS)
assert all(master.groupby('name_kr').size().reindex(STATIONS)==460)
assert all(spatial.groupby('지점').size().reindex(STATIONS)==100)
print('자료 확인:',len(master),'지점·일 / 공간분석',len(spatial),'지점·일')
display(terrain)


## 1. 공간분석 날짜 선정의 재검증
기존 보고서 방법: 세 지점 모두 **ASOS ≥ 1mm**, 하루 GPM 30분 자료 **48개**, 지정 태풍 가능 영향기간 제외.

제외일: 2020-08-09~10, 2021-08-23~24, 2023-08-08~11. 분석일은 공통 100일이며 3지점 300개 기록이다.


In [ ]:
blocked = set(pd.date_range('2020-08-09','2020-08-10').strftime('%Y-%m-%d'))
blocked |= set(pd.date_range('2021-08-23','2021-08-24').strftime('%Y-%m-%d'))
blocked |= set(pd.date_range('2023-08-08','2023-08-11').strftime('%Y-%m-%d'))
eligible = master.loc[(master['asos_mm_filled']>=1)&(master['half_hour_count']==48)]
candidate_dates = eligible.groupby('date_kst')['stn'].nunique()
selected_dates = set(candidate_dates[candidate_dates==3].index) - blocked
observed_dates = set(spatial['날짜'])
print('공통 강수일 후보:',sum(candidate_dates==3),'일 → 제외 후:',len(selected_dates),'일')
assert selected_dates == observed_dates, '공간 분석 CSV의 날짜 목록과 선정 기준이 다릅니다.'
merged = spatial.merge(master, left_on=['날짜','지점'], right_on=['date_kst','name_kr'],validate='1:1')
assert np.allclose(merged['ASOS_mm'],merged['asos_mm_filled'],atol=1e-6)
assert np.allclose(merged['GPM_mm'],merged['gpm_cal_mm'],atol=1e-6)
assert np.allclose(merged['GPM_절대편차_mm'],np.abs(merged['GPM_mm']-merged['ASOS_mm']),atol=1e-5)
print('선정 날짜, 기존 GPM·ASOS 입력값 및 편차 대조 완료')


## 2. 지형지표와 전체 460일 오차 비교
**고도 표준편차**는 부산 126.32m, 울산 64.06m, 창원 123.48m. **평균 경사도**와 **고지대(≥150m) 비율**도 함께 비교한다.

지형은 시간에 따라 변하지 않는 지점별 3개 값이므로 여기서는 세 지점의 서술적 순위 비교만 수행하고 지형–오차 상관계수를 제시하지 않는다.


In [ ]:
df=master.copy()
df['error_mm'] = df['gpm_cal_mm']-df['asos_mm_filled']
station_summary = (df.groupby('name_kr').agg(n=('error_mm','size'),
                        Bias_mm=('error_mm','mean'),MAE_mm=('error_mm',lambda x:x.abs().mean()),
                        RMSE_mm=('error_mm',lambda x:np.sqrt(np.mean(np.square(x)))))
                   .reindex(STATIONS).reset_index().rename(columns={'name_kr':'지점'}))
terrain_compare=terrain.merge(station_summary,on='지점',validate='1:1')
display(terrain_compare.round(3))
assert np.isclose(terrain_compare.loc[terrain_compare['지점']=='부산','고도표준편차_m'].iloc[0],126.32,atol=.01)
terrain_compare.to_csv(OUT/'03_terrain_station_comparison.csv',index=False,encoding='utf-8-sig')


In [ ]:
fig,ax=plt.subplots(figsize=(7.5,5))
ax.scatter(terrain_compare['고도표준편차_m'],terrain_compare['MAE_mm'],s=115)
for _,row in terrain_compare.iterrows():
    ax.annotate(NAME_EN[row['지점']],(row['고도표준편차_m'],row['MAE_mm']),xytext=(6,5),textcoords='offset points')
ax.set(xlabel='Elevation standard deviation (m)',ylabel='460-day MAE (mm)',title='Subgoal 3 | Terrain relief vs. station MAE')
ax.grid(alpha=.25);fig.tight_layout()
fig.savefig(OUT/'03_terrain_mae_comparison.png',dpi=180);plt.show()


## 3. 공통 100일 강수 공간변동성과 편차
500m 육지 격자점(부산 366, 울산 353, 창원 332개)의 원시 일강수량으로 산출된 **공간 CV = 공간표준편차 / 공간평균 × 100**을 사용한다.

CV와 각 날짜의 GPM–ASOS **절대편차**에 대한 스피어만 ρ를 각 지점 100일로 계산한다.


In [ ]:
assert (spatial.groupby('지점')['유효격자수'].nunique()==1).all()
print('지점별 육지 500m 격자점 수:',spatial.groupby('지점')['유효격자수'].first().to_dict())
cv_recomputed = spatial['공간표준편차_mm']/spatial['500m_격자평균_mm']*100
assert np.allclose(cv_recomputed,spatial['공간변동계수_pct'],atol=.005)
def group_stats(g):
    cv=g['공간변동계수_pct'].to_numpy(dtype=float)
    ae=g['GPM_절대편차_mm'].to_numpy(dtype=float)
    rho,p=spearmanr(cv,ae)
    return pd.Series({'n_days':len(g),'ASOS_mean_mm':g['ASOS_mm'].mean(),
           'GPM_mean_mm':g['GPM_mm'].mean(), 'Bias_mean_mm':g['GPM_Bias_mm'].mean(),
           'MAE_mm':ae.mean(),'spatial_CV_mean_pct':cv.mean(),
           'Spearman_rho_CV_AE':rho,'Spearman_p_descriptive':p,
           'highlow_abs_diff_mean_mm':g['고저강수량_절대차_mm'].mean()})
spatial_summary = pd.DataFrame([{'지점':name,**group_stats(spatial[spatial['지점']==name]).to_dict()} for name in STATIONS])
display(spatial_summary.round(4))
assert np.allclose(spatial_summary['spatial_CV_mean_pct'],[15.9340,13.8778,10.2975],atol=.002)
assert np.allclose(spatial_summary['Spearman_rho_CV_AE'],[-.160192,-.233579,-.233003],atol=.001)
spatial_summary.to_csv(OUT/'03_spatial_cv_station_summary.csv',index=False,encoding='utf-8-sig')


In [ ]:
fig,ax=plt.subplots(figsize=(8,5.5))
for site in STATIONS:
    g=spatial[spatial['지점']==site]
    ax.scatter(g['공간변동계수_pct'],g['GPM_절대편차_mm'],s=27,alpha=.65,
               label=f'{NAME_EN[site]} (rho={spearmanr(g["공간변동계수_pct"],g["GPM_절대편차_mm"]).statistic:.3f})')
ax.set(xlabel='Within-grid rainfall CV (%)',ylabel='Absolute error |GPM - ASOS| (mm)',
       title='Subgoal 3 | Rainfall heterogeneity vs. absolute error (100 days)')
ax.legend();ax.grid(alpha=.2);fig.tight_layout()
fig.savefig(OUT/'03_cv_absolute_error_scatter.png',dpi=180);plt.show()


## 4. 보조 비교: 공통 30mm 이상 강수일 22일과 강수 구간
이 분석은 추가 탐색으로서 본 보고서에서 100일 분석 결과와 구분하여 표기한다. 강수 구간은 **각 지점의 ASOS 값**으로 분류한다.


In [ ]:
all_heavy_dates = set(spatial.groupby('날짜')['ASOS_mm'].min().loc[lambda v:v>=30].index)
assert len(all_heavy_dates)==22
rows=[]
for site in STATIONS:
    for label,g in [('100일_공통1mm이상',spatial[spatial['지점']==site]),
                    ('22일_공통30mm이상',spatial[(spatial['지점']==site)&spatial['날짜'].isin(all_heavy_dates)])]:
        rho=spearmanr(g['공간변동계수_pct'],g['GPM_절대편차_mm']).statistic
        rows.append({'지점':site,'분석범위':label,'일수':len(g),'Spearman_rho':rho})
scope_rho=pd.DataFrame(rows);display(scope_rho.round(4))
assert np.allclose(scope_rho.query('분석범위=="22일_공통30mm이상"')['Spearman_rho'],[.4217956,.2433653,-.0084698],atol=.001)
spatial['rain_bin'] = pd.cut(spatial['ASOS_mm'], bins=[1,10,30,np.inf],right=False,
                             labels=['1–<10 mm','10–<30 mm','>=30 mm'])
assert spatial['rain_bin'].notna().all()
bin_sum=(spatial.groupby(['지점','rain_bin'],observed=True).agg(n=('날짜','size'),
        mean_ASOS_mm=('ASOS_mm','mean'),MAE_mm=('GPM_절대편차_mm','mean'),
        CV_mean_pct=('공간변동계수_pct','mean')).reset_index())
display(bin_sum.round(3))
scope_rho.to_csv(OUT/'03_scope_spearman.csv',index=False,encoding='utf-8-sig')
bin_sum.to_csv(OUT/'03_exploratory_rain_bin.csv',index=False,encoding='utf-8-sig')


In [ ]:
# 지점별로 강수 규모와 CV·절대편차의 평균 변화 비교
for metric,filename,ylabel in [
    ('CV_mean_pct','03_cv_by_rain_bin.png','Mean within-grid CV (%)'),
    ('MAE_mm','03_mae_by_rain_bin.png','Mean absolute error (mm)')]:
    pv=bin_sum.pivot(index='rain_bin',columns='지점',values=metric).reindex(columns=STATIONS)
    ax=pv.rename(columns=NAME_EN).plot(kind='bar',figsize=(8,4.8),rot=0)
    ax.set(title=f'Subgoal 3 | {ylabel} by observed rainfall',xlabel='ASOS rainfall class',ylabel=ylabel)
    ax.grid(axis='y',alpha=.2);ax.legend(title='Station');plt.tight_layout();plt.savefig(OUT/filename,dpi=180);plt.show()
print('재계산 및 그림 저장 완료:',OUT.resolve())


## 5. 원자료를 추가 확보했을 때의 확장 위치

- 강수 공간 CV의 원시 단계부터 다시 계산하려면 각 날짜·지점별 **육지 500m 격자점의 실제 `rn_day` 값과 고도**, 데이터 품질 마스크가 필요하다.
- 기존 보고서에서는 KMA 원자료 23:55 KST 기준 파일을 사용했다. 원시 500m 격자파일과 정확한 복원/단위 처리법이 공개되어야 완전한 원자료 단계 재현이 가능하다.
- 여기서는 중간 산출표의 `공간표준편차_mm / 500m_격자평균_mm × 100` 계산을 재확인하고, 날짜 선정·GPM/ASOS 매칭 및 통계를 처음부터 계산했다.
- 기존 별도 Colab의 DEM 코드에는 Earth Engine 인증과 DEM의 원투영·GPM 원시 격자 경계 일치성 확인이 필요하다. **본 노트북의 지형 요약값은 보고서에 사용한 검증된 산출표**를 기준으로 유지한다.

공식 자료: [Copernicus DEM GLO-30 on Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_DEM_GLO30), [GPM-IMERG V07](https://developers.google.com/earth-engine/datasets/catalog/NASA_GPM_L3_IMERG_V07).
